# Taller 2 (AWS Academy): ETL con Apache Spark y Delta Lake en EMR

Implementarás el mismo flujo ETL Medallion (Bronze → Silver → Gold) del Taller Databricks,  
pero sobre **Amazon Web Services** usando **EMR** (Elastic MapReduce) como motor de Spark  
y **S3** como sistema de almacenamiento del data lake.

## Diferencias clave respecto a Databricks
| Concepto | Databricks | AWS EMR |
|---|---|---|
| Almacenamiento | DBFS (`/tmp/...`) | S3 (`s3://bucket/...`) |
| Operaciones de archivos | `dbutils.fs` | `boto3` o AWS CLI (`aws s3`) |
| Visualización en notebook | `display(df)` | `df.show()` / `df.limit(n).toPandas()` |
| Delta Lake | Pre-instalado | Configurado al crear el cluster |
| SparkSession | Pre-inicializada | Pre-inicializada (kernel PySpark) |

## ¿Qué es AWS Academy?

AWS Academy es un programa educativo de Amazon que otorga acceso gratuito a servicios de AWS  
a estudiantes a través de sus instituciones. Incluye un **Learner Lab** con créditos de ~$100 USD  
y acceso a los servicios necesarios para este taller: **S3**, **EMR** y **EMR Notebooks**.

---
## 0. Configuración del entorno en AWS Academy

Sigue estos pasos **una sola vez** antes de ejecutar el notebook.

---

### Paso 1 — Acceder al Learner Lab de AWS Academy

Tu institución debe estar inscrita en AWS Academy. Si no has recibido la invitación, consulta con tu docente.

**Windows / macOS / Linux:**
1. Revisa tu correo institucional: deberías haber recibido una invitación a **AWS Academy**
2. Acepta la invitación y crea tu cuenta en https://www.awsacademy.com/
3. Inicia sesión → navega a tu curso asignado
4. Abre el módulo **"Learner Lab"** (o **"AWS Academy Learner Lab"**)
5. Haz clic en **"Start Lab"** — espera 1-2 minutos hasta que el indicador cambie a verde (●)
6. Haz clic en **"AWS"** (enlace verde) para abrir la consola de AWS en una nueva pestaña

> Monitorea tu saldo restante en el panel del Learner Lab. El saldo inicial es ~$100 USD.  
> El lab se pausa automáticamente tras un período de inactividad para conservar créditos.

---

### Paso 2 — Crear un bucket en S3

S3 es el equivalente al DBFS de Databricks: el almacenamiento distribuido del data lake.

1. En la consola de AWS, escribe **"S3"** en la barra de búsqueda y selecciónalo
2. Haz clic en **"Create bucket"**
3. **Bucket name:** elige un nombre único globalmente, por ejemplo `taller2-spark-tunombre`  
   *(los nombres de bucket S3 son únicos en todo AWS)*
4. **AWS Region:** usa la región asignada por el Learner Lab (generalmente `us-east-1`)
5. Deja **Block all public access** activado (opción por defecto)
6. Haz clic en **"Create bucket"**

> Anota el nombre exacto del bucket. Lo usarás en la variable `BUCKET_NAME` del notebook.

---

### Paso 3 — Crear el cluster de EMR con Spark y Delta Lake

EMR es el servicio de Spark administrado de AWS, equivalente a Dataproc en GCP.

1. En la consola, busca **"EMR"** y selecciónalo
2. En el panel izquierdo, selecciona **"Clusters"** → haz clic en **"Create cluster"**
3. Configura las opciones principales:
   - **Cluster name:** `taller2-cluster`
   - **Amazon EMR release:** `emr-6.15.0` o la versión más reciente disponible (6.x o superior)
   - **Application bundle:** selecciona **"Spark"** (incluye Spark + JupyterEnterpriseGateway)
4. En **"Instance configuration"**:
   - **Primary node:** `m5.xlarge` (suficiente para el taller)
   - **Core nodes:** cambia la cantidad a `0` (solo nodo principal, más económico)
5. En **"Software settings"**, pega la siguiente configuración JSON para activar Delta Lake:

```json
[
  {
    "Classification": "spark-defaults",
    "Properties": {
      "spark.jars.packages": "io.delta:delta-core_2.12:2.4.0",
      "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension",
      "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    }
  }
]
```

6. En **"Cluster logs"**: activa **"Publish cluster-specific logs to Amazon S3"** y selecciona tu bucket
7. En **"Security configuration"**: en el Learner Lab generalmente se puede dejar el EC2 key pair en blanco
8. Haz clic en **"Create cluster"** y espera 5-10 minutos hasta que el estado sea **"Waiting"** (color verde)

> **Costo estimado:** un cluster `m5.xlarge` consume aproximadamente $0.20 USD/hora de tus créditos.  
> Recuerda **terminarlo** (Terminate) cuando termines el taller para no gastar créditos innecesariamente.

---

### Paso 4 — Crear un EMR Notebook y conectarlo al cluster

EMR Notebooks (parte de EMR Studio) es el equivalente al workspace de notebooks de Databricks.

1. En el panel de EMR, selecciona **"Notebooks"** en el menú izquierdo
2. Haz clic en **"Create notebook"**
3. Configura:
   - **Notebook name:** `taller2-notebook`
   - **Cluster:** selecciona **"Choose an existing cluster"** → selecciona `taller2-cluster`
   - **AWS Service Role:** deja la opción por defecto del Learner Lab
   - **Notebook location:** selecciona tu bucket S3
4. Haz clic en **"Create notebook"**
5. Una vez creado (estado **"Ready"**), haz clic en **"Open in JupyterLab"**

---

### Paso 5 — Importar este notebook y configurar el kernel

1. En JupyterLab, ve a **File → Upload Files** y sube el archivo `guia_aws_academy.ipynb`
2. Abre el notebook
3. **Verifica que el kernel sea PySpark** (esquina superior derecha del notebook)
   - Si no dice PySpark, haz clic → selecciona **PySpark**
4. En la primera celda de código, reemplaza `TU_BUCKET` con el nombre de tu bucket S3

> En EMR Notebooks con kernel PySpark, `spark` y `sc` están disponibles automáticamente.  
> No necesitas crear SparkSession ni instalar nada adicional.

---
## Arquitectura del taller

```
  DATOS SINTETICOS
  (generados con Spark)
          |
          v
 +----------------+
 |    BRONZE      |  s3://bucket/taller2/bronze/
 |   Delta Lake   |  Datos crudos, sin transformar
 +----------------+
          |
          v
 +----------------+
 |    SILVER      |  s3://bucket/taller2/silver/
 |   Delta Lake   |  Datos limpios y enriquecidos
 +----------------+
          |
          v
 +----------------+
 |     GOLD       |  s3://bucket/taller2/gold/
 |   Delta Lake   |  Metricas para consumo analitico
 +----------------+
```

**Almacenamiento:** Amazon S3  
**Formato:** Delta Lake (Parquet + transaction log)  
**Motor de procesamiento:** Apache Spark en Amazon EMR

In [ ]:
# =============================================================
# CONFIGURACION — ACTUALIZA ESTE VALOR ANTES DE CONTINUAR
# =============================================================

BUCKET_NAME = "TU_BUCKET"  # <-- reemplaza con el nombre de tu bucket S3

# Rutas en S3 (equivalentes al DBFS de Databricks)
BASE_PATH   = f"s3://{BUCKET_NAME}/taller2"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH   = f"{BASE_PATH}/gold"

# Limpiar ejecuciones anteriores
# aws s3 rm reemplaza a dbutils.fs.rm en el contexto de S3
import subprocess
subprocess.run(
    ["aws", "s3", "rm", f"s3://{BUCKET_NAME}/taller2/", "--recursive"],
    capture_output=True
)
print("Directorio anterior eliminado (si existia).")

print(f"\nRutas configuradas:")
print(f"  Bronze : {BRONZE_PATH}")
print(f"  Silver : {SILVER_PATH}")
print(f"  Gold   : {GOLD_PATH}")
print(f"\nSpark version: {spark.version}")

---
## Parte 1 — Bronze: Ingesta de datos crudos

Generamos 1.000 transacciones de ventas sintéticas y las guardamos como **Delta Lake en S3**.

In [ ]:
# =============================================================
# 1.1 — GENERAR DATOS SINTETICOS
# =============================================================

from pyspark.sql import functions as F

SEED = 42

arr_categorias = F.array([F.lit(c) for c in ["Electronica", "Muebles", "Ropa", "Alimentos", "Libros"]])
arr_ciudades   = F.array([F.lit(c) for c in ["Bogota", "Medellin", "Cali", "Barranquilla", "Cartagena"]])
arr_pagos      = F.array([F.lit(p) for p in ["tarjeta_credito", "tarjeta_debito", "efectivo", "transferencia"]])
arr_productos  = F.array([F.lit(p) for p in [
    "Laptop Pro", "Silla Ergonomica", "Camiseta Algodon", "Arroz Premium",
    "Python Avanzado", "Monitor 4K", "Escritorio Pie", "Jean Clasico",
    "Aceite Oliva", "SQL para Todos", "Auriculares BT", "Estante Madera",
    "Zapatos Cuero", "Leche Entera", "Historia IA"
]])

df_bronze_raw = (
    spark.range(1, 1001)
    .select(
        F.col("id"),
        F.rand(SEED).alias("r1"),
        F.rand(SEED + 1).alias("r2"),
        F.rand(SEED + 2).alias("r3"),
        F.rand(SEED + 3).alias("r4"),
        F.rand(SEED + 4).alias("r5"),
        F.rand(SEED + 5).alias("r6"),
        F.rand(SEED + 6).alias("r7"),
    )
    .select(
        F.concat(F.lit("TXN-"), F.lpad(F.col("id").cast("string"), 5, "0")).alias("transaction_id"),
        F.concat(F.lit("C"), F.lpad(((F.col("r1") * 200).cast("int") + 1).cast("string"), 4, "0")).alias("customer_id"),
        arr_productos[(F.col("r2") * 15).cast("int")].alias("product_name"),
        arr_categorias[(F.col("r3") * 5).cast("int")].alias("category"),
        ((F.col("r4") * 9).cast("int") + 1).alias("quantity"),
        F.when(F.col("r5") < 0.03, F.lit(-99.0))
         .otherwise(F.round(F.col("r5") * 1995 + 5, 2)).alias("unit_price"),
        F.date_add(F.lit("2024-01-01").cast("date"), (F.col("r6") * 365).cast("int")).alias("transaction_date"),
        arr_ciudades[(F.col("r1") * 5).cast("int")].alias("store_city"),
        arr_pagos[(F.col("r2") * 4).cast("int")].alias("payment_method"),
        F.when(F.col("r7") > 0.9, F.lit(None).cast("double"))
         .otherwise((F.col("r4") * 4).cast("int") * 0.05).alias("discount_pct"),
    )
)

print(f"Registros generados: {df_bronze_raw.count():,}")
df_bronze_raw.printSchema()
df_bronze_raw.show(5, truncate=False)

In [ ]:
# =============================================================
# 1.2 — GUARDAR BRONZE EN S3 COMO DELTA LAKE
# Delta Lake escribe en S3 exactamente igual que en DBFS.
# El transaction log (_delta_log/) se almacena en S3.
# =============================================================

(
    df_bronze_raw
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{BRONZE_PATH}/ventas/")
)

print(f"Tabla Bronze guardada en: {BRONZE_PATH}/ventas/")

# Ver archivos en S3 (aws s3 ls reemplaza a dbutils.fs.ls)
result = subprocess.run(
    ["aws", "s3", "ls", f"s3://{BUCKET_NAME}/taller2/bronze/ventas/"],
    capture_output=True, text=True
)
print("\nArchivos en Bronze (S3):")
print(result.stdout)

In [ ]:
# =============================================================
# 1.3 — EXPLORAR BRONZE
# En EMR Notebooks usamos .show() en lugar de display()
# =============================================================

df_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/ventas/")

print(f"Total registros  : {df_bronze.count():,}")
print(f"Total columnas   : {len(df_bronze.columns)}")

print("\n=== Nulos por columna ===")
df_bronze.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_bronze.columns
]).show()

print(f"Precios invalidos (< 0): {df_bronze.filter(F.col('unit_price') < 0).count()}")

print("\n=== Distribucion por categoria ===")
df_bronze.groupBy("category").count().orderBy("count", ascending=False).show()

# Muestra en formato tabla Pandas (para datasets pequenos)
print("\n=== Muestra (primeras 5 filas) ===")
df_bronze.limit(5).toPandas()

---
## Parte 2 — Silver: Limpieza y transformación

In [ ]:
# =============================================================
# 2.1 — TRANSFORMACION SILVER
# Leer Bronze desde S3 → limpiar → enriquecer → guardar Silver en S3
# =============================================================

df_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/ventas/")

df_silver = (
    df_bronze
    .filter(F.col("unit_price") > 0)
    .fillna({"discount_pct": 0.0})
    .withColumn("total_bruto",    F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("total_descuento",F.round(F.col("total_bruto") * F.col("discount_pct"), 2))
    .withColumn("total_neto",     F.round(F.col("total_bruto") - F.col("total_descuento"), 2))
    .withColumn("year",           F.year("transaction_date"))
    .withColumn("month",          F.month("transaction_date"))
    .withColumn("month_name",     F.date_format("transaction_date", "MMMM"))
    .withColumn("day_of_week",    F.date_format("transaction_date", "EEEE"))
    .withColumn("processed_at",   F.current_timestamp())
)

print(f"Bronze  : {df_bronze.count():,} registros")
print(f"Silver  : {df_silver.count():,} registros")
print(f"Eliminados: {df_bronze.count() - df_silver.count():,}")

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("year", "month")
    .save(f"{SILVER_PATH}/ventas/")
)

print(f"\nSilver guardada en: {SILVER_PATH}/ventas/")

In [ ]:
# =============================================================
# 2.2 — VERIFICAR SILVER
# =============================================================

df_silver_check = spark.read.format("delta").load(f"{SILVER_PATH}/ventas/")

print("=== Nulos en columnas criticas (deben ser 0) ===")
df_silver_check.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["unit_price", "discount_pct", "total_neto"]
]).show()

print("=== Muestra de datos Silver ===")
df_silver_check.select(
    "transaction_id", "customer_id", "category",
    "quantity", "unit_price", "discount_pct", "total_neto",
    "year", "month"
).show(10)

---
## Parte 3 — Gold: Métricas y agregaciones

In [ ]:
# =============================================================
# 3.1 — VENTAS POR CATEGORIA Y MES
# =============================================================

df_silver = spark.read.format("delta").load(f"{SILVER_PATH}/ventas/")

gold_cat_mes = (
    df_silver
    .groupBy("year", "month", "month_name", "category")
    .agg(
        F.count("transaction_id").alias("num_transacciones"),
        F.sum("quantity").alias("unidades_vendidas"),
        F.round(F.sum("total_bruto"), 2).alias("ingresos_brutos"),
        F.round(F.sum("total_neto"), 2).alias("ingresos_netos"),
        F.round(F.avg("unit_price"), 2).alias("precio_promedio"),
    )
    .orderBy("year", "month", F.col("ingresos_netos").desc())
)

gold_cat_mes.show(15)
gold_cat_mes.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/ventas_categoria_mes/")
print(f"Guardado en: {GOLD_PATH}/ventas_categoria_mes/")

In [ ]:
# =============================================================
# 3.2 — TOP CLIENTES + SEGMENTACION
# =============================================================

gold_clientes = (
    df_silver
    .groupBy("customer_id")
    .agg(
        F.count("transaction_id").alias("num_compras"),
        F.round(F.sum("total_neto"), 2).alias("gasto_total"),
        F.round(F.avg("total_neto"), 2).alias("gasto_promedio"),
        F.countDistinct("category").alias("categorias_distintas"),
    )
    .withColumn(
        "segmento",
        F.when(F.col("gasto_total") >= 5000, "VIP")
         .when(F.col("gasto_total") >= 2000, "Premium")
         .when(F.col("gasto_total") >= 500,  "Regular")
         .otherwise("Ocasional")
    )
    .orderBy(F.col("gasto_total").desc())
)

print("=== Top 15 clientes ===")
gold_clientes.show(15)

print("=== Distribucion de segmentos ===")
gold_clientes.groupBy("segmento").count().orderBy("count", ascending=False).show()

gold_clientes.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/top_clientes/")
print(f"Guardado en: {GOLD_PATH}/top_clientes/")

In [ ]:
# =============================================================
# 3.3 — ANALISIS POR METODO DE PAGO (Spark SQL)
# =============================================================

df_silver.createOrReplaceTempView("silver_ventas")

gold_pagos = spark.sql("""
    SELECT
        payment_method,
        COUNT(*)                           AS num_transacciones,
        ROUND(SUM(total_neto), 2)          AS ingresos_netos,
        ROUND(AVG(total_neto), 2)          AS ticket_promedio,
        ROUND(AVG(discount_pct) * 100, 1)  AS descuento_promedio_pct
    FROM silver_ventas
    GROUP BY payment_method
    ORDER BY ingresos_netos DESC
""")

gold_pagos.show()
gold_pagos.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/metodos_pago/")
print(f"Guardado en: {GOLD_PATH}/metodos_pago/")

---
## Parte 4 — Delta Lake: Capacidades avanzadas

Delta Lake es un **formato de tabla abierto** que funciona de forma idéntica sobre S3 que sobre DBFS.  
El transaction log (`_delta_log/`) se almacena directamente en el bucket S3.

In [ ]:
# =============================================================
# 4.1 — TRANSACTION LOG: DESCRIBE HISTORY
# =============================================================

from delta.tables import DeltaTable

silver_dt = DeltaTable.forPath(spark, f"{SILVER_PATH}/ventas/")

print("=== Historial de la tabla Silver ===")
silver_dt.history().select(
    "version", "timestamp", "operation", "numOutputRows"
).show(truncate=False)

# Ver el transaction log en S3
result = subprocess.run(
    ["aws", "s3", "ls", f"s3://{BUCKET_NAME}/taller2/silver/ventas/_delta_log/"],
    capture_output=True, text=True
)
print("\nArchivos del transaction log en S3:")
print(result.stdout)

In [ ]:
# =============================================================
# 4.2 — CREAR NUEVA VERSION CON APPEND
# =============================================================

df_nuevos = df_silver.limit(50).withColumn("processed_at", F.current_timestamp())

(
    df_nuevos
    .write
    .format("delta")
    .mode("append")
    .save(f"{SILVER_PATH}/ventas/")
)

print("Append completado. Versiones disponibles:")
silver_dt.history().select("version", "timestamp", "operation").show()

In [ ]:
# =============================================================
# 4.3 — TIME TRAVEL: LEER VERSION ANTERIOR
# Funciona igual en S3 que en DBFS gracias al transaction log
# =============================================================

df_actual = spark.read.format("delta").load(f"{SILVER_PATH}/ventas/")

df_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(f"{SILVER_PATH}/ventas/")
)

print(f"Version actual  : {df_actual.count():,} registros")
print(f"Version 0       : {df_v0.count():,} registros")
print(f"Diferencia      : {df_actual.count() - df_v0.count():,} registros")

In [ ]:
# =============================================================
# 4.4 — OPTIMIZE: COMPACTAR ARCHIVOS EN S3
# =============================================================

spark.sql(f"OPTIMIZE delta.`{SILVER_PATH}/ventas/`")

print("OPTIMIZE completado.")
silver_dt.history().select("version", "operation").show(5)

---
## Parte 5 — Ejercicios integradores

Los mismos ejercicios del Taller Databricks, ahora ejecutándose sobre AWS.

**E1** — Producto más vendido (en unidades) por categoría.

In [ ]:
# Tu respuesta aquí


**E2** — Ingresos netos y tasa de descuento promedio por ciudad (`store_city`). Ordena de mayor a menor ingreso.

In [ ]:
# Tu respuesta aquí


**E3** — Tabla Gold `tendencia_semanal` con: semana del año, número de transacciones, ingresos netos y ticket promedio. Guárdala en `{GOLD_PATH}/tendencia_semanal/` como Delta.

In [ ]:
# Tu respuesta aquí


---
## Soluciones de referencia

In [ ]:
# --- E1 ---
# from pyspark.sql import Window
# w = Window.partitionBy("category").orderBy(F.col("total_unidades").desc())
# (
#     df_silver.groupBy("category", "product_name")
#     .agg(F.sum("quantity").alias("total_unidades"))
#     .withColumn("rank", F.rank().over(w))
#     .filter(F.col("rank") == 1).drop("rank")
#     .orderBy("category").show()
# )

# --- E2 ---
# (
#     df_silver.groupBy("store_city")
#     .agg(
#         F.round(F.avg("discount_pct") * 100, 1).alias("descuento_prom_pct"),
#         F.round(F.sum("total_neto"), 2).alias("ingresos_netos")
#     )
#     .orderBy(F.col("ingresos_netos").desc()).show()
# )

# --- E3 ---
# gold_semanal = (
#     df_silver
#     .withColumn("week", F.weekofyear("transaction_date"))
#     .groupBy("year", "week")
#     .agg(
#         F.count("transaction_id").alias("num_transacciones"),
#         F.round(F.sum("total_neto"), 2).alias("ingresos_netos"),
#         F.round(F.avg("total_neto"), 2).alias("ticket_promedio")
#     )
#     .orderBy("year", "week")
# )
# gold_semanal.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/tendencia_semanal/")
# gold_semanal.show()

---
## Limpieza de recursos

**Importante:** para preservar tus créditos del Learner Lab, termina el cluster cuando termines el taller.

In [ ]:
# Eliminar datos del taller del bucket S3
result = subprocess.run(
    ["aws", "s3", "rm", f"s3://{BUCKET_NAME}/taller2/", "--recursive"],
    capture_output=True, text=True
)
print(f"Datos eliminados de s3://{BUCKET_NAME}/taller2/")

### Terminar el cluster de EMR

1. Ve a la consola de AWS → **EMR** → **Clusters**
2. Selecciona `taller2-cluster`
3. Haz clic en **"Terminate"** y confirma

> Un cluster en estado **"Waiting"** sigue consumiendo créditos aunque no esté procesando datos.

### Pausar el Learner Lab (si ya no vas a continuar)

1. Regresa al portal de AWS Academy
2. Haz clic en **"End Lab"** para pausar el entorno y detener el consumo de créditos